In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
import sqlite3

conn = sqlite3.connect('repos.db')
c = conn.cursor()

c.execute('''
    CREATE TABLE IF NOT EXISTS repos (
        name TEXT,
        language TEXT,
        stars INTEGER
    )
''')
conn.commit()

page = 1

while True:
    url = f"https://github.com/orgs/google/repositories?page={page}"
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    try:
        res = requests.get(url, headers=headers)
        res.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching page {page}: {e}")
        break

    soup = BeautifulSoup(res.text, "html.parser")

    script = soup.find("script", {"data-target": "react-app.embeddedData"})
    if script is None:
        print(f"Could not find embedded data script on page {page}.")
        break

    try:
        data = json.loads(script.string)
        repos = data["payload"]["orgReposPageRoute"]["repositories"]
    except (KeyError, json.JSONDecodeError) as e:
        print(f"Could not parse data on page {page}: {e}")
        break

    if not repos:
        print(f"No more repositories found on page {page}.")
        break

    for repo in repos:
        name = repo.get("name", "N/A")
        lang_info = repo.get("primaryLanguage") or {}
        language = lang_info.get("name", "N/A")
        stars = repo.get("starsCount", 0)
        
        c.execute("INSERT INTO repos (name, language, stars) VALUES (?, ?, ?)", (name, language, stars))

    conn.commit()
    print(f"Scraped page {page}, found {len(repos)} repositories.")
    page += 1
    # time.sleep(1) # Be a good citizen and don't spam the server

conn.close()

# Display the saved data
conn = sqlite3.connect('repos.db')
c = conn.cursor()

print("\n--- Data from repos.db ---")
for row in c.execute("SELECT * FROM repos ORDER BY stars DESC"):
    print(row)

conn.close()

Scraped page 1, found 30 repositories.
Scraped page 2, found 30 repositories.
Scraped page 3, found 30 repositories.
Scraped page 4, found 30 repositories.
Scraped page 5, found 30 repositories.
Scraped page 6, found 30 repositories.
Scraped page 7, found 30 repositories.
Scraped page 8, found 30 repositories.
Scraped page 9, found 30 repositories.
Scraped page 10, found 30 repositories.
Scraped page 11, found 30 repositories.
Scraped page 12, found 30 repositories.
Scraped page 13, found 30 repositories.
Scraped page 14, found 30 repositories.
Scraped page 15, found 30 repositories.
Scraped page 16, found 30 repositories.
Scraped page 17, found 30 repositories.
Scraped page 18, found 30 repositories.
Scraped page 19, found 30 repositories.
Scraped page 20, found 30 repositories.
Scraped page 21, found 30 repositories.
Scraped page 22, found 30 repositories.
Scraped page 23, found 30 repositories.
Scraped page 24, found 30 repositories.
Scraped page 25, found 30 repositories.
Scraped p